# Actual Facility and District Insights

This notebook focuses on planning findings from the cleaned district dataset: medical deserts, care gaps, trust gaps, and best-care signals. The earlier cleaning notebook explains metadata quality; this one uses the cleaned fields to answer where to act.


## What Counts as Ground Truth?

- NFHS district health indicators are treated as the most reliable district-level need signal in this dataset.
- India Post PIN-to-district mapping is treated as the geographic bridge, but it still has ambiguity and naming drift.
- Facility rows are not ground truth. They are FDR web-extracted claims with source text and URLs. They become an observed supply signal only after join, geography, source, and evidence-quality checks.
- The “medical desert” labels are triage categories, not final policy truth. They should drive verification, re-survey, referral planning, or deployment decisions.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "output" / "data"
district = pd.read_csv(DATA_DIR / "district_health_facility_cleaned.csv")
facility = pd.read_csv(DATA_DIR / "facility_health_cleaned.csv", low_memory=False)
summary = json.loads((DATA_DIR / "actual_insights_summary.json").read_text())
district.shape, facility.shape


## Decision Categories

These categories mirror the planning matrix:

- `real_desert_candidate`: high need, low trustworthy observed supply.
- `phantom_desert_or_verification_gap`: facilities exist, but trust/evidence is weak.
- `supply_record_quality_problem`: supply exists but records are contradicted, geo-invalid, or suspicious.
- `referral_or_capacity_candidate`: lower need, better trust, and observed supply that may support referrals while gaps are fixed.
- `mixed_or_monitor`: not cleanly classified.


In [ ]:
district["planning_category"].value_counts().to_frame("districts")


![Planning categories](../../output/plots/actual_insights/planning_category_counts.png)


## Medical Deserts and Worst Care Gaps


![Top care gaps](../../output/plots/actual_insights/top_care_gaps.png)


![Medical desert candidates](../../output/plots/actual_insights/top_medical_desert_candidates.png)


In [ ]:
cols = [
    "state_ut", "district_name", "planning_category", "observed_facility_rows",
    "trustworthy_supply_rows", "health_need_score", "care_gap_score",
    "district_medical_desert_priority_score", "sample_facility_names"
]
district.sort_values("care_gap_score", ascending=False)[cols].head(20)


## Least Trustworthy Regions


![Least trustworthy](../../output/plots/actual_insights/least_trustworthy_regions.png)


In [ ]:
cols = [
    "state_ut", "district_name", "planning_category", "observed_facility_rows",
    "trustworthy_supply_rate", "needs_human_review_rate",
    "contradicted_or_geo_invalid_rate", "trust_gap_score",
    "predominant_join_uncertainty"
]
district.sort_values("trust_gap_score", ascending=False)[cols].head(20)


## Best Care Signals


![Best care signals](../../output/plots/actual_insights/best_care_signal_regions.png)


In [ ]:
cols = [
    "state_ut", "district_name", "observed_facility_rows",
    "trustworthy_supply_rows", "health_need_score",
    "district_data_quality_score", "best_care_signal_score",
    "sample_facility_names"
]
district.sort_values("best_care_signal_score", ascending=False)[cols].head(20)


## Need vs Supply


![Need vs supply](../../output/plots/actual_insights/need_vs_trustworthy_supply.png)


## Claimed Care Signals in Top Gap Districts


![Care signal heatmap](../../output/plots/actual_insights/care_signal_heatmap_top_gaps.png)


In [ ]:
care_cols = [
    "state_ut", "district_name", "care_gap_score",
    "maternity_signal_rate", "emergency_signal_rate",
    "diagnostic_signal_rate", "ncd_signal_rate", "trustworthy_supply_rate"
]
district.sort_values("care_gap_score", ascending=False)[care_cols].head(30)


## Facility Evidence Scatter


![Facility trust scatter](../../output/plots/actual_insights/facility_trustworthy_supply_map_scatter.png)


## Practical Interpretation

The highest-scoring “real desert” candidates should not immediately become capital build decisions. Use the category to choose the next action:

- Real desert candidate: verify with local knowledge, then plan mobile unit or permanent capacity.
- Phantom desert or verification gap: send a verification/re-survey team first.
- Supply record quality problem: fix records and geocoding before supply planning.
- Referral or capacity candidate: consider interim routing or referral networks while high-gap districts are reviewed.


In [ ]:
pd.DataFrame(summary['rankings']['top_care_gaps']).head(15)
